## Import libraries, load data, and initialize the nlp tools

In [ ]:
import os, spacy
import pandas as pd
from collections import defaultdict
from typing import List, Dict, Optional
from transformers import pipeline



nlp = spacy.load("es_core_news_md")
sentiment_model = pipeline(
        task="sentiment-analysis", 
        model="pysentimiento/robertuito-sentiment-analysis",
        device=0
    )

answers = pd.read_excel(
    os.path.join('..', 'questions', 'dummy-data.xlsx'),
    index_col=0,
    sheet_name='Questionario'
)
questions = pd.read_excel(
    os.path.join('..', 'questions', 'dummy-data.xlsx'),
    sheet_name='Preguntas'
)

c:\Users\User\miniconda3\envs\kpi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9200.44it/s]


In [2]:
def sentencizer(answers: pd.DataFrame) -> pd.DataFrame:
    """Splits the answers into sentences and creates a DataFrame with the following columns:
    - student: the name of the student
    - question: the name of the question
    - sentence_number: the number of the sentence
    - sentence: the text of the sentence
    
    Args:
        answers (pd.DataFrame): the DataFrame with the answers of the students
    
    Returns:
        pd.DataFrame: the DataFrame with the sentences of the answers
    """
    corpus = []
    for student in answers.index:
        for question in answers.columns:
            doc = nlp(answers[question].loc[student])
            sentences = [sent.text.replace('\n', '') for sent in doc.sents]
            df = pd.DataFrame({
                'student': student,
                'question': question,
                'sentence_number': list(range(len(sentences))),
                'sentence': sentences
            })
            corpus.append(df)
    return pd.concat(corpus).reset_index(drop=True)

def sentiment_analyzer(corpus: pd.DataFrame) -> pd.DataFrame:
    """Analyzes the sentiment of the sentences and adds it to the DataFrame with the following columns:
    - sentiment_label: the label of the sentiment (positive, negative, neutral)
    - sentiment_score: the score of the sentiment
    
    Args:
        corpus (pd.DataFrame): the DataFrame with the sentences of the answers. This dataframe is the output of the sentencizer function
    
    Returns:
        pd.DataFrame: the DataFrame with the sentiments of the sentences
    """
    sentiment = sentiment_model(corpus.sentence.tolist())
    sentiment = pd.DataFrame(sentiment).rename(
        {'label': 'sentiment_label', 'score': 'sentiment_score'},
        axis=1
    )
    return pd.concat([corpus, sentiment], axis=1)

In [ ]:
def classify_sentence(
    sentence: str,
    program_keywords: Optional[List[str]] = None
) -> Dict:
    """
    Classify if a sentence is about the student (self) or the program.
    
    Args:
        sentence: Individual sentence to classify
        program_keywords: List of program-specific terms
    
    Returns:
        Classification result with scores and evidence
    """
    if program_keywords is None:
        program_keywords = [
            'programa', 'pertene-ser', 'perteneser', 'docentes', 'profesores',
            'tutores', 'acompañamiento', 'apoyo', 'apoyos', 'grupo', 'equipo',
            'adaptaciones', 'estrategias', 'técnicas', 'herramientas', 'espacio',
            'espacios', 'materiales', 'evaluación', 'introdujo', 'brindó',
            'ofreció', 'ofrecieron', 'enseñó', 'promovió', 'permitió', 'ayudaron',
            'ayudó', 'escucharon', 'validaron', 'respaldo', 'seguimiento', 'reuniones',
            'retroalimentación', 'orientación', 'actividades', 'fomentaron', 'facilitó',
            'reconocidos', 'aceptación', 'validación', 'cercanía', 'personalizado',
            'personalización', 'comunicación', 'oportunidad', 'universidad', 'recursos',
            'accesibilidad', 'talleres', 'sensibilización', 'limitaciones', 'potencial',
            'ajustes', 'diagnóstico', 'plan', 'capacitación', 'formación', 'canales',
            'solicitudes', 'disponible', 'virtuales', 'participación', 'bienestar',
            'escucha', 'psicológicos', 'transformación', 'iniciativa', 'propósito',
            'impacto', 'tutorías', 'estrategia', 'fomentar'
        ]
    
    doc = nlp(sentence)
    
    scores = {
        'student': 0,
        'program': 0
    }
    
    evidence = defaultdict(list)
    
    # Track if sentence has mixed content
    has_student_markers = False
    has_program_markers = False
    
    for token in doc:
        # STUDENT MARKERS: First person pronouns and possessives
        if token.pos_ in ["PRON", "DET"]:
            if token.text.lower() in ['mi', 'mis', 'me', 'yo', 'mí']:
                scores['student'] += 2
                evidence['student'].append(f"1st_person_pronoun: '{token.text}'")
                has_student_markers = True
        
        # STUDENT MARKERS: First person verbs
        if token.pos_ in ["VERB", "AUX"]:
            morph = token.morph.to_dict()
            if morph.get('Person') == '1':
                scores['student'] += 1
                evidence['student'].append(f"1st_person_verb: '{token.lemma_}'")
                has_student_markers = True
        
        # PROGRAM MARKERS: Program-related keywords
        lemma_lower = token.lemma_.lower()
        text_lower = token.text.lower()
        
        if lemma_lower in program_keywords or text_lower in program_keywords:
            scores['program'] += 3
            evidence['program'].append(f"keyword: '{token.text}'")
            has_program_markers = True
        
        # PROGRAM MARKERS: Third person subjects (especially nouns)
        if token.dep_ == "nsubj":
            morph = token.morph.to_dict()
            # Third person subjects
            if morph.get('Person') == '3' or token.pos_ in ["NOUN", "PROPN"]:
                # Check if it's program-related
                if lemma_lower in program_keywords or text_lower in program_keywords:
                    scores['program'] += 2
                    evidence['program'].append(f"program_subject: '{token.text}'")
                    has_program_markers = True
    
    # Determine classification
    if scores['student'] > scores['program']:
        classification = "STUDENT"
    elif scores['program'] > scores['student']:
        classification = "PROGRAM"
    elif scores['student'] == scores['program'] and scores['student'] > 0:
        classification = "MIXED"
    else:
        classification = "UNCLEAR"
    
    # Flag mixed sentences (both student and program elements)
    is_mixed = has_student_markers and has_program_markers
    
    return {
        'classification': classification,
        'scores': scores,
        'evidence': dict(evidence),
        'is_mixed': is_mixed,
        'sentence': sentence.strip()
    }


def classify_sentences_batch(
    sentences: List[str],
    program_keywords: Optional[List[str]] = None
) -> List[Dict]:
    """
    Classify a list of pre-segmented sentences.
    
    Args:
        sentences: List of individual sentences
        program_keywords: Optional custom program keywords
    
    Returns:
        List of classification results
    """
    results = []
    
    for sentence in sentences:
        if sentence.strip():  # Skip empty sentences
            result = classify_sentence(sentence, program_keywords)
            results.append(result)
    
    return results

In [6]:
corpus = sentencizer(answers)
corpus = sentiment_analyzer(corpus)
results = pd.DataFrame(classify_sentences_batch(corpus.sentence.tolist()))
results.to_excel(os.path.join('..', 'output', 'classification_results.xlsx'), index=False)
corpus = corpus.assign(sentence_subject=results.classification)
corpus.to_excel(os.path.join('..', 'output', 'stundent_support_corpus.xlsx'), index=False)

In [8]:
df = pd.read_excel(os.path.join('..', 'output', 'stundent_support_corpus.xlsx'), index_col=0)
df = df[(df.sentiment_label=='NEG')&(df.sentence_subject=='PROGRAM')]
docs = df.sentence.tolist()
docs

['Antes, veía a los profesores como figuras distantes, difíciles de acercar.',
 'Pasé de sentirme aislado y ajeno a la comunidad universitaria, a experimentar aceptación, reconocimiento y vínculos reales.',
 'Gracias a este seguimiento, logré evitar que los problemas se acumularan y se convirtieran en crisis.',
 'Entre las actividades que menos me sirvieron, podría mencionar algunas dinámicas grupales demasiado abiertas, donde no había roles claros ni estructura definida.',
 'Aunque la intención era fomentar la integración, la falta de claridad generaba más ansiedad que beneficios.',
 'Las actividades menos útiles fueron aquellas que carecían de estructura, aunque con ajustes podrían convertirse en herramientas valiosas.',
 'Desde el inicio, los docentes mostraron interés en comprender mi situación particular, en lugar de aplicar un esquema genérico.',
 'Uno de ellos es la sobrecarga de estímulos en ciertos ambientes universitarios, como espacios muy concurridos o actividades sociales 

## Topic modeling for program negative things